### Dependencies and Configuration

#### Phase 2 

move to next phase


In [3]:
# Cell 1.0: Write Self-Contained fa_evyolo_modules.py to Disk
from pathlib import Path

MODULE_SOURCE = '''"""Custom PyTorch modules for the FA-EvYOLO architecture."""
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision


class SpectralAmplitudeNorm2d(nn.Module):
    """
    Pillar 1B: Spectral Amplitude Normalization with Phase-Coherence Gate (SAN-PCG).
    Decouples domain-variant illumination/glare amplitude from structural phase
    in the 2D Fourier domain while maintaining FP32/FP16 compatibility under AMP.
    """

    def __init__(self, c1: int = 64, c2: int = None, eps: float = 1e-4) -> None:
        super().__init__()
        c_out = c1 if c2 is None else c2
        self.c = c_out
        self.eps = eps

        self.proj = nn.Identity() if c1 == c_out else nn.Conv2d(c1, c_out, kernel_size=1, bias=False)
        self.amp_gamma = nn.Parameter(torch.ones(1, c_out, 1, 1))
        self.amp_beta = nn.Parameter(torch.zeros(1, c_out, 1, 1))
        self.mix_logit = nn.Parameter(torch.tensor(0.0))

        self.phase_gate = nn.Sequential(
            nn.Conv2d(c_out, c_out, kernel_size=3, padding=1, groups=c_out, bias=False),
            nn.BatchNorm2d(c_out),
            nn.Conv2d(c_out, c_out, kernel_size=1, bias=True),
            nn.Sigmoid(),
        )
        self.post_norm = nn.BatchNorm2d(c_out)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        orig_dtype = x.dtype
        x = self.proj(x)
        _, _, h, w = x.shape

        # Fourier transform operations require FP32 because cuFFT only supports
        # power-of-two dimensions for FP16, and feature maps (e.g. 80x80, 40x40, 20x20)
        # are not powers of two.
        with torch.amp.autocast("cuda", enabled=False):
            x_fp32 = torch.nan_to_num(x.float(), nan=0.0, posinf=1e4, neginf=-1e4)
            fft_x = torch.fft.rfft2(x_fp32, norm="ortho")
            real = fft_x.real
            imag = fft_x.imag

            amp = torch.sqrt(real * real + imag * imag + self.eps)
            phase_real = real / amp
            phase_imag = imag / amp
            phase_unit = torch.complex(phase_real, phase_imag)

            log_amp = torch.log(amp.clamp(min=self.eps))
            mean = log_amp.mean(dim=(-2, -1), keepdim=True)
            std = log_amp.std(dim=(-2, -1), keepdim=True, unbiased=False).clamp(min=1e-2)
            z = ((log_amp - mean) / std).clamp(-4.0, 4.0)

            gamma = self.amp_gamma.float().clamp(-1.5, 1.5)
            beta = self.amp_beta.float().clamp(-1.5, 1.5)
            norm_amp = torch.exp(z * gamma + beta)

            alpha = torch.sigmoid(self.mix_logit.float())
            blended_amp = alpha * norm_amp + (1.0 - alpha) * amp

            fft_norm = torch.complex(blended_amp * phase_real, blended_amp * phase_imag)
            x_spec = torch.fft.irfft2(fft_norm, s=(h, w), norm="ortho")
            x_phase = torch.fft.irfft2(phase_unit, s=(h, w), norm="ortho")

        w_dtype = self.phase_gate[0].weight.dtype
        x_spec = x_spec.to(w_dtype)
        x_phase = x_phase.to(w_dtype)

        gate = self.phase_gate(x_phase)
        out = self.act(self.post_norm(x_spec * gate))

        alpha_act = torch.sigmoid(self.mix_logit).to(orig_dtype)
        res = x + alpha_act * out.to(orig_dtype)

        return torch.nan_to_num(res, nan=0.0, posinf=1e4, neginf=-1e4)


class RetinexFourierTransform(nn.Module):
    """
    Pillar 1A: Multi-Scale Retinex Reflectance Normalization & Fourier Amplitude Mix (Retinex-FAM).
    """

    def __init__(self, retinex_weight: float = 0.25, fam_prob: float = 0.50, fam_strength: float = 0.30) -> None:
        super().__init__()
        self.retinex_weight = retinex_weight
        self.fam_prob = fam_prob
        self.fam_strength = fam_strength

    @torch.no_grad()
    def forward(self, img: torch.Tensor, training: bool = False) -> torch.Tensor:
        orig_dtype = img.dtype
        x = img.to(torch.float32).clamp(0.0, 1.0)

        illum_fine = F.avg_pool2d(x, kernel_size=7, stride=1, padding=3)
        illum_coarse = F.avg_pool2d(x, kernel_size=15, stride=1, padding=7)
        illum = 0.5 * (illum_fine + illum_coarse)

        reflectance = torch.log(x + 1e-3) - torch.log(illum + 1e-3)
        r_min = reflectance.amin(dim=(-2, -1), keepdim=True)
        r_max = reflectance.amax(dim=(-2, -1), keepdim=True)
        reflectance_norm = (reflectance - r_min) / (r_max - r_min + 1e-5)

        x = (1.0 - self.retinex_weight) * x + self.retinex_weight * reflectance_norm

        if training and x.shape[0] > 1 and torch.rand(1, device=x.device).item() < self.fam_prob:
            b, c, h, w = x.shape
            fft_x = torch.fft.rfft2(x, norm="ortho")
            amp = torch.abs(fft_x)
            phase = torch.angle(fft_x)

            perm = torch.randperm(b, device=x.device)
            lam = torch.empty((b, 1, 1, 1), device=x.device).uniform_(0.0, self.fam_strength)
            mixed_amp = (1.0 - lam) * amp + lam * amp[perm]

            fft_mixed = torch.polar(mixed_amp, phase)
            x = torch.fft.irfft2(fft_mixed, s=(h, w), norm="ortho")

        return x.clamp(0.0, 1.0).to(orig_dtype)


class EvidentialRegularizer(nn.Module):
    """
    Pillar 2: Unified Evidential Subjective Logic Regularizer (Ev-CLS & Ev-DFL)
    and Single-Pass Epistemic/Aleatoric Uncertainty Extractor.
    """

    def __init__(self, reg_max: int = 16, kl_weight: float = 0.05) -> None:
        super().__init__()
        self.reg_max = reg_max
        self.kl_weight = kl_weight

    def compute_ev_cls_penalty(self, pred_scores: torch.Tensor, target_scores: torch.Tensor) -> torch.Tensor:
        pred_fp32 = pred_scores.to(torch.float32).clamp(-15.0, 15.0)
        target_fp32 = target_scores.to(torch.float32).clamp(0.0, 1.0)

        e_pos = F.softplus(pred_fp32)
        e_neg = F.softplus(-pred_fp32)
        alpha = e_pos + 1.0
        beta = e_neg + 1.0
        strength = alpha + beta

        p_mean = alpha / strength
        bayes_var = (alpha * beta) / ((strength ** 2) * (strength + 1.0))
        err_sq = (target_fp32 - p_mean) ** 2

        bg_mask = (1.0 - target_fp32).clamp(0.0, 1.0)
        kl_bg = bg_mask * (e_pos ** 2) / (strength + 1e-5)

        return (err_sq + bayes_var + self.kl_weight * kl_bg).mean()

    def compute_ev_dfl_penalty(
        self, pred_dist: torch.Tensor, target_bins: torch.Tensor, fg_mask: torch.Tensor
    ) -> torch.Tensor:
        if fg_mask.sum() == 0:
            return pred_dist.sum() * 0.0

        fg_dist = pred_dist[fg_mask].to(torch.float32).view(-1, self.reg_max).clamp(-15.0, 15.0)
        alpha = F.softplus(fg_dist) + 1.0
        strength = alpha.sum(dim=-1, keepdim=True)
        probs = alpha / strength

        bins = torch.arange(self.reg_max, device=fg_dist.device, dtype=torch.float32).view(1, -1)
        expected_coord = (probs * bins).sum(dim=-1)
        second_moment = (probs * (bins ** 2)).sum(dim=-1)
        intrinsic_var = (second_moment - expected_coord ** 2).clamp(min=0.0)
        dirichlet_var = intrinsic_var / (strength.squeeze(-1) + 1.0)

        t_flat = target_bins.to(torch.float32).view(-1).clamp(0.0, float(self.reg_max - 1) - 1e-3)
        coord_err = F.smooth_l1_loss(expected_coord, t_flat, reduction="none")

        return (0.15 * coord_err + 0.10 * dirichlet_var).mean()

    @staticmethod
    @torch.no_grad()
    def extract_box_uncertainty(cls_logits: torch.Tensor, dfl_logits: torch.Tensor, reg_max: int = 16) -> dict:
        cls_fp32 = cls_logits.to(torch.float32).clamp(-10.0, 10.0)
        e_pos = 2.0 * torch.exp(cls_fp32)
        u_cls = (2.0 / (e_pos + 2.0)).clamp(0.0, 1.0)

        dfl_reshaped = dfl_logits.to(torch.float32).view(*dfl_logits.shape[:-1], 4, reg_max)
        p_dfl = F.softmax(dfl_reshaped, dim=-1)
        ent_dfl = -(p_dfl * torch.log(p_dfl.clamp(min=1e-8))).sum(dim=-1) / float(np.log(float(reg_max)))

        alpha_loc = F.softplus(dfl_reshaped) + 1.0
        vac_dfl = (float(reg_max) / alpha_loc.sum(dim=-1)).clamp(0.0, 1.0)
        u_loc = (0.5 * ent_dfl.mean(dim=-1) + 0.5 * vac_dfl.mean(dim=-1)).clamp(0.0, 1.0)

        if u_cls.ndim > u_loc.ndim:
            u_cls = u_cls.squeeze(-1)

        u_total = (0.55 * u_cls + 0.45 * u_loc).clamp(0.0, 1.0)
        return {"u_cls": u_cls, "u_loc": u_loc, "u_total": u_total}


class ReflectionUncertaintyNMS(nn.Module):
    """
    Pillar 3: Reflection-Aware Uncertainty-Modulated Non-Maximum Suppression (RU-NMS).
    """

    def __init__(self, gamma_u: float = 0.35, gamma_g: float = 0.65, iou_thres: float = 0.65) -> None:
        super().__init__()
        self.gamma_u = gamma_u
        self.gamma_g = gamma_g
        self.iou_thres = iou_thres

    @staticmethod
    @torch.no_grad()
    def compute_box_glare_density(img_tensor: torch.Tensor, boxes_xyxy: torch.Tensor) -> torch.Tensor:
        if boxes_xyxy.numel() == 0:
            return torch.zeros((0,), device=boxes_xyxy.device, dtype=torch.float32)

        _, h, w = img_tensor.shape
        c_max = img_tensor.amax(dim=0)
        c_min = img_tensor.amin(dim=0)
        glare_map = ((c_max >= 0.85) & ((c_max - c_min) <= 0.20)).to(torch.float32)

        integral = F.pad(glare_map, (1, 0, 1, 0), mode="constant", value=0.0).cumsum(dim=0).cumsum(dim=1)

        x1 = boxes_xyxy[:, 0].clamp(0, w - 1).long()
        y1 = boxes_xyxy[:, 1].clamp(0, h - 1).long()
        x2 = torch.maximum(boxes_xyxy[:, 2].clamp(1, w).long(), x1 + 1)
        y2 = torch.maximum(boxes_xyxy[:, 3].clamp(1, h).long(), y1 + 1)

        area = ((x2 - x1) * (y2 - y1)).to(torch.float32).clamp(min=1.0)
        glare_sum = integral[y2, x2] - integral[y1, x2] - integral[y2, x1] + integral[y1, x1]
        return (glare_sum / area).clamp(0.0, 1.0)

    @torch.no_grad()
    def modulate_and_filter(
        self,
        boxes_xyxy: torch.Tensor,
        scores: torch.Tensor,
        vacuity: torch.Tensor,
        img_tensor: torch.Tensor,
        conf_thres: float = 0.0,
        use_glare: bool = True,
        use_vacuity: bool = True,
    ) -> tuple:
        if boxes_xyxy.numel() == 0:
            return boxes_xyxy, scores, torch.zeros((0,), dtype=torch.long, device=boxes_xyxy.device)

        glare_density = (
            self.compute_box_glare_density(img_tensor, boxes_xyxy)
            if use_glare
            else torch.zeros_like(scores, dtype=torch.float32)
        )
        u_flat = vacuity.view(-1).clamp(0.0, 1.0) if use_vacuity else torch.zeros_like(scores, dtype=torch.float32)
        s_flat = scores.view(-1).clamp(0.0, 1.0)

        damping = torch.exp(-self.gamma_u * u_flat - self.gamma_g * glare_density * u_flat)
        mod_scores = (s_flat * damping).clamp(0.0, 1.0)

        valid_mask = mod_scores >= conf_thres
        if not valid_mask.any():
            return (
                torch.zeros((0, 4), device=boxes_xyxy.device, dtype=boxes_xyxy.dtype),
                torch.zeros((0,), device=scores.device, dtype=scores.dtype),
                torch.zeros((0,), dtype=torch.long, device=boxes_xyxy.device),
            )

        valid_idx = torch.where(valid_mask)[0]
        nms_keep = torchvision.ops.nms(boxes_xyxy[valid_idx], mod_scores[valid_idx], self.iou_thres)
        keep_idx = valid_idx[nms_keep]
        return boxes_xyxy[keep_idx], mod_scores[keep_idx], keep_idx
'''

Path("fa_evyolo_modules.py").write_text(MODULE_SOURCE, encoding="utf-8")
print("fa_evyolo_modules.py written to disk.")


fa_evyolo_modules.py written to disk.


In [4]:
# Cell 1.1: Dependencies, Logging, and Path Verification
import os
import random
import shutil
import logging
from pathlib import Path
import cv2
import numpy as np
import yaml

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

RAW_PATHS = {
    "marine_litter": Path(r"D:\Journal-Project\Datasets\Floating Marine Litter.v3i.yolo26"),
    "floating_waste": Path(r"D:\Journal-Project\Datasets\Floating Waste.v10i.yolo26"),
    "flow_img": Path(r"D:\Journal-Project\Datasets\flow-img.v1i.yolo26"),
}
OUTPUT_ROOT = Path(r"D:\Journal-Project\Datasets\Q1_Harmonized_Benchmark")

def verify_paths(paths_dict: dict) -> bool:
    valid = True
    for name, path in paths_dict.items():
        if not path.exists():
            logger.error(f"Missing directory for {name}: {path.resolve()}")
            valid = False
    return valid

if verify_paths(RAW_PATHS):
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    logger.info(f"Cell 1.1 executed: Paths verified and output directory ready at {OUTPUT_ROOT}")
else:
    logger.critical("Cell 1.1 executed: Path verification failed.")

2026-10-30 08:57:04,838 - INFO - Cell 1.1 executed: Paths verified and output directory ready at D:\Journal-Project\Datasets\Q1_Harmonized_Benchmark


In [5]:
# Cell 1.2: Continuous Reflection Severity Index (RSI) Engine and YOLO Label Sanitizer
def compute_continuous_rsi(img_bgr: np.ndarray) -> float:
    """Computes continuous Reflection Severity Index (RSI) in [0.0, 1.0]."""
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s, v = cv2.split(hsv)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)

    f_glint = float(np.mean((v >= 220) & (s <= 50)))

    sobel_x = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    grad_mag = np.sqrt(sobel_x ** 2 + sobel_y ** 2)
    bright_mask = v > 170
    e_streak = min(1.0, float(np.mean(grad_mag[bright_mask]) / 255.0)) if np.any(bright_mask) else 0.0

    p95, p50 = np.percentile(v, 95), np.percentile(v, 50)
    contrast_tail = float(np.clip((p95 - p50) / 255.0, 0.0, 1.0))

    return 0.50 * min(1.0, f_glint * 6.0) + 0.30 * e_streak + 0.20 * contrast_tail


def sanitize_and_copy(img_path: Path, lbl_path: Path, out_split_dir: Path) -> list:
    """Remaps valid object class IDs to 0, clamps coordinates to [0.0, 1.0], and copies files."""
    if not lbl_path.exists():
        return []

    clean_lines = []
    with open(lbl_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            try:
                xc, yc, bw, bh = map(float, parts[1:5])
            except ValueError:
                continue

            x1 = max(0.0, min(1.0, xc - bw / 2.0))
            y1 = max(0.0, min(1.0, yc - bh / 2.0))
            x2 = max(0.0, min(1.0, xc + bw / 2.0))
            y2 = max(0.0, min(1.0, yc + bh / 2.0))

            new_bw = x2 - x1
            new_bh = y2 - y1
            if new_bw > 1e-4 and new_bh > 1e-4:
                new_xc = x1 + new_bw / 2.0
                new_yc = y1 + new_bh / 2.0
                clean_lines.append(f"0 {new_xc:.6f} {new_yc:.6f} {new_bw:.6f} {new_bh:.6f}")

    if not clean_lines:
        return []

    shutil.copy2(img_path, out_split_dir / "images" / img_path.name)
    with open(out_split_dir / "labels" / f"{img_path.stem}.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(clean_lines))

    return clean_lines

logger.info("Cell 1.2 executed: Continuous RSI Engine and Label Sanitizer defined.")

2026-10-30 08:57:13,421 - INFO - Cell 1.2 executed: Continuous RSI Engine and Label Sanitizer defined.


In [6]:
# Cell 1.3: Harmonize Datasets, Split flow_img (80/10/10), and Build Quantile-Balanced RSI Strata
def build_q1_benchmark():
    # 1. Sanitize and copy raw splits
    for name, raw_dir in RAW_PATHS.items():
        if not raw_dir.exists():
            logger.warning(f"Dataset not found at {raw_dir}, skipping.")
            continue

        logger.info(f"Harmonizing dataset: {name.upper()}")
        out_dataset_dir = OUTPUT_ROOT / name

        for split in ["train", "valid", "test"]:
            in_split = raw_dir / split
            if not in_split.exists():
                continue

            std_split = "val" if split == "valid" else split
            out_split = out_dataset_dir / std_split
            (out_split / "images").mkdir(parents=True, exist_ok=True)
            (out_split / "labels").mkdir(parents=True, exist_ok=True)

            img_list = [p for p in (in_split / "images").glob("*.*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]
            logger.info(f"  Processing {len(img_list)} images in {split} split.")
            for img_path in img_list:
                lbl_path = in_split / "labels" / f"{img_path.stem}.txt"
                sanitize_and_copy(img_path, lbl_path, out_split)

        yaml_data = {
            "path": str(out_dataset_dir.resolve()),
            "train": "train/images",
            "val": "val/images",
            "test": "test/images",
            "names": {0: "floating_litter"},
        }
        with open(out_dataset_dir / f"{name}.yaml", "w", encoding="utf-8") as f:
            yaml.dump(yaml_data, f, sort_keys=False)

    # 2. Deterministic 80/10/10 split for flow_img if val/test are missing
    flow_dir = OUTPUT_ROOT / "flow_img"
    flow_train_imgs = sorted(list((flow_dir / "train" / "images").glob("*.*")))
    flow_test_dir = flow_dir / "test" / "images"

    if not flow_test_dir.exists() or len(list(flow_test_dir.glob("*.*"))) == 0:
        logger.info(f"Splitting flow_img ({len(flow_train_imgs)} train images) into 80% train, 10% val, 10% test.")
        rng = random.Random(42)
        shuffled = flow_train_imgs.copy()
        rng.shuffle(shuffled)

        n_total = len(shuffled)
        n_val = int(0.10 * n_total)
        n_test = int(0.10 * n_total)

        val_files = shuffled[:n_val]
        test_files = shuffled[n_val : n_val + n_test]

        for split_name, file_list in [("val", val_files), ("test", test_files)]:
            s_img_dir = flow_dir / split_name / "images"
            s_lbl_dir = flow_dir / split_name / "labels"
            s_img_dir.mkdir(parents=True, exist_ok=True)
            s_lbl_dir.mkdir(parents=True, exist_ok=True)

            for img_path in file_list:
                lbl_path = flow_dir / "train" / "labels" / f"{img_path.stem}.txt"
                if lbl_path.exists():
                    shutil.move(str(img_path), str(s_img_dir / img_path.name))
                    shutil.move(str(lbl_path), str(s_lbl_dir / lbl_path.name))

    # 3. Stratify val and test splits across all 3 datasets into balanced tertiles (Low_RSI, Med_RSI, High_RSI)
    for name in ["marine_litter", "floating_waste", "flow_img"]:
        ds_dir = OUTPUT_ROOT / name
        for s_type in ["val", "test"]:
            split_img_dir = ds_dir / s_type / "images"
            split_lbl_dir = ds_dir / s_type / "labels"
            if not split_img_dir.exists():
                continue

            strata_root = ds_dir / f"rsi_strata_{s_type}"
            if strata_root.exists():
                shutil.rmtree(strata_root)

            scored_items = []
            for img_path in sorted(split_img_dir.glob("*.*")):
                img_bgr = cv2.imread(str(img_path))
                if img_bgr is not None:
                    scored_items.append((compute_continuous_rsi(img_bgr), img_path))

            if not scored_items:
                continue

            scored_items.sort(key=lambda x: x[0])
            n = len(scored_items)
            t1, t2 = n // 3, (2 * n) // 3
            tertiles = {
                "Low_RSI": scored_items[:t1],
                "Med_RSI": scored_items[t1:t2],
                "High_RSI": scored_items[t2:],
            }

            counts_log = []
            for rsi_level, items in tertiles.items():
                r_img_dir = strata_root / rsi_level / "images"
                r_lbl_dir = strata_root / rsi_level / "labels"
                r_img_dir.mkdir(parents=True, exist_ok=True)
                r_lbl_dir.mkdir(parents=True, exist_ok=True)

                for _, img_path in items:
                    lbl_path = split_lbl_dir / f"{img_path.stem}.txt"
                    if lbl_path.exists():
                        shutil.copy2(img_path, r_img_dir / img_path.name)
                        shutil.copy2(lbl_path, r_lbl_dir / lbl_path.name)

                s_yaml = {
                    "path": str(ds_dir.resolve()),
                    "train": str(r_img_dir.relative_to(ds_dir.resolve())),
                    "val": str(r_img_dir.relative_to(ds_dir.resolve())),
                    "test": str(r_img_dir.relative_to(ds_dir.resolve())),
                    "names": {0: "floating_litter"},
                }
                with open(ds_dir / f"{name}_{s_type}_{rsi_level}.yaml", "w", encoding="utf-8") as f:
                    yaml.dump(s_yaml, f, sort_keys=False)
                counts_log.append(f"{rsi_level}={len(items)}")

            logger.info(f"[{name.upper()} - {s_type}] Balanced RSI Strata: {', '.join(counts_log)}")

    logger.info("Phase 1 complete: Datasets harmonized, split, and RSI-stratified.")

build_q1_benchmark()

2026-10-30 08:57:21,182 - INFO - Harmonizing dataset: MARINE_LITTER
2026-10-30 08:57:21,204 - INFO -   Processing 7424 images in train split.
2026-10-30 08:58:30,177 - INFO -   Processing 1058 images in valid split.
2026-10-30 08:58:39,754 - INFO -   Processing 528 images in test split.
2026-10-30 08:58:44,314 - INFO - Harmonizing dataset: FLOATING_WASTE
2026-10-30 08:58:44,374 - INFO -   Processing 6687 images in train split.
2026-10-30 09:00:06,747 - INFO -   Processing 638 images in valid split.
2026-10-30 09:00:14,615 - INFO -   Processing 319 images in test split.
2026-10-30 09:00:18,495 - INFO - Harmonizing dataset: FLOW_IMG
2026-10-30 09:00:18,564 - INFO -   Processing 5992 images in train split.
2026-10-30 09:01:38,504 - INFO - Splitting flow_img (5992 train images) into 80% train, 10% val, 10% test.
2026-10-30 09:01:55,361 - INFO - [MARINE_LITTER - val] Balanced RSI Strata: Low_RSI=237, Med_RSI=237, High_RSI=237
2026-10-30 09:02:03,323 - INFO - [MARINE_LITTER - test] Balanced 

In [11]:
# Cell 2.1: Train Baseline YOLO11n on marine_litter and Evaluate Zero-Shot Matrix
import pandas as pd
from ultralytics import YOLO

DATASETS = ["marine_litter", "floating_waste", "flow_img"]
RSI_STRATA = ["Low_RSI", "Med_RSI", "High_RSI"]

def run_baseline_training_and_evaluation():
    source_data = "marine_litter"
    source_yaml = OUTPUT_ROOT / source_data / f"{source_data}.yaml"

    saved_weights = Path(r"D:\Journal-Project\runs\detect\runs\Baseline_YOLO11n\train_marine_litter\weights\best.pt")
    if not saved_weights.exists():
        logger.info("Training Baseline YOLO11n on MARINE_LITTER for 100 epochs...")
        base_model = YOLO("yolo11n.pt")
        base_model.train(
            data=str(source_yaml),
            epochs=100,
            imgsz=640,
            batch=16,
            project="runs/Baseline_YOLO11n",
            name="train_marine_litter",
            exist_ok=True,
            seed=42,
            deterministic=True,
            verbose=False,
        )
    else:
        logger.info(f"Found existing trained Baseline YOLO11n checkpoint at: {saved_weights}")

    model = YOLO(str(saved_weights))
    results_records = []

    for target_data in DATASETS:
        is_in_domain = source_data == target_data
        domain_type = "In-Domain" if is_in_domain else "Out-of-Domain (Zero-Shot)"
        logger.info(f"Evaluating Baseline_YOLO11n on Target: {target_data.upper()} ({domain_type})")

        target_yaml = OUTPUT_ROOT / target_data / f"{target_data}.yaml"
        if target_yaml.exists():
            metrics = model.val(data=str(target_yaml), split="test", verbose=False)
            results_records.append({
                "Model": "Baseline_YOLO11n",
                "Train_Source": source_data,
                "Test_Target": target_data,
                "Domain_Shift": domain_type,
                "RSI_Glare_Level": "ALL",
                "mAP_50": round(float(metrics.box.map50), 4),
                "mAP_50_95": round(float(metrics.box.map), 4),
            })

        for rsi in RSI_STRATA:
            rsi_yaml = OUTPUT_ROOT / target_data / f"{target_data}_test_{rsi}.yaml"
            if rsi_yaml.exists():
                rsi_metrics = model.val(data=str(rsi_yaml), split="test", verbose=False)
                results_records.append({
                    "Model": "Baseline_YOLO11n",
                    "Train_Source": source_data,
                    "Test_Target": target_data,
                    "Domain_Shift": domain_type,
                    "RSI_Glare_Level": rsi,
                    "mAP_50": round(float(rsi_metrics.box.map50), 4),
                    "mAP_50_95": round(float(rsi_metrics.box.map), 4),
                })

    df = pd.DataFrame(results_records)
    out_csv = OUTPUT_ROOT / "Table1_CrossDataset_RSI_Matrix.csv"
    df.to_csv(out_csv, index=False)
    print("\n--- BASELINE CROSS-DATASET & RSI PERFORMANCE MATRIX ---")
    print(df.to_string(index=False))
    return df

baseline_eval_df = run_baseline_training_and_evaluation()

2026-10-30 09:12:36,827 - INFO - Training Baseline YOLO11n on MARINE_LITTER for 100 epochs...


New https://pypi.org/project/ultralytics/8.4.193 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.182  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Journal-Project\Datasets\Q1_Harmonized_Benchmark\marine_litter\marine_litter.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01,

2026-10-30 10:11:40,029 - INFO - Evaluating Baseline_YOLO11n on Target: MARINE_LITTER (In-Domain)


Ultralytics 8.4.182  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11n summary (fused): 101 layers, 2,597,051 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.4±0.1 ms, read: 100.2±24.3 MB/s, size: 48.8 KB)
val: Scanning D:\Journal-Project\Datasets\Q1_Harmonized_Benchmark\marine_litter\test\labels... 348 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 348/348 872.0it/s 0.4s0.1s
val: New cache created: D:\Journal-Project\Datasets\Q1_Harmonized_Benchmark\marine_litter\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 1.4it/s 15.9s0.1s
                   all        348       1571      0.884      0.857      0.925      0.593
Speed: 1.1ms preprocess, 3.0ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to D:\Journal-Project\runs\detect\val
Ultralytics 8.4.182  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 La

2026-10-30 10:12:44,234 - INFO - Evaluating Baseline_YOLO11n on Target: FLOATING_WASTE (Out-of-Domain (Zero-Shot))


Ultralytics 8.4.182  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11n summary (fused): 101 layers, 2,597,051 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.2±0.0 ms, read: 107.8±16.5 MB/s, size: 41.3 KB)
val: Scanning D:\Journal-Project\Datasets\Q1_Harmonized_Benchmark\floating_waste\test\labels... 306 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 306/306 907.9it/s 0.3s0.1s
val: New cache created: D:\Journal-Project\Datasets\Q1_Harmonized_Benchmark\floating_waste\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 1.4it/s 14.4s<0.2s
                   all        306        896      0.419      0.285      0.227      0.074
Speed: 0.6ms preprocess, 2.0ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to D:\Journal-Project\runs\detect\val-5
Ultralytics 8.4.182  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 40

2026-10-30 10:13:13,537 - INFO - Evaluating Baseline_YOLO11n on Target: FLOW_IMG (Out-of-Domain (Zero-Shot))


Ultralytics 8.4.182  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11n summary (fused): 101 layers, 2,597,051 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.3±0.1 ms, read: 238.3±52.3 MB/s, size: 128.5 KB)
val: Scanning D:\Journal-Project\Datasets\Q1_Harmonized_Benchmark\flow_img\test\labels... 599 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 599/599 829.8it/s 0.7s0.0s
val: New cache created: D:\Journal-Project\Datasets\Q1_Harmonized_Benchmark\flow_img\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 2.2it/s 17.2s0.1s
                   all        599       1570      0.578      0.376      0.399      0.147
Speed: 1.1ms preprocess, 3.1ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to D:\Journal-Project\runs\detect\val-9
Ultralytics 8.4.182  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GP

In [ ]:
# Cell 3.1: Register Custom Modules, Build fa_evyolo.yaml, and Verify GPU Forward/Backward Pass
import copy
import importlib
import yaml
import torch
from pathlib import Path
from ultralytics import YOLO
import ultralytics.nn.modules as ul_modules
import ultralytics.nn.tasks as ul_tasks

import fa_evyolo_modules
importlib.reload(fa_evyolo_modules)
from fa_evyolo_modules import (
    SpectralAmplitudeNorm2d,
    RetinexFourierTransform,
    EvidentialRegularizer,
    ReflectionUncertaintyNMS,
)

# Register SpectralAmplitudeNorm2d in Ultralytics namespace
setattr(ul_modules, "SpectralAmplitudeNorm2d", SpectralAmplitudeNorm2d)
setattr(ul_tasks, "SpectralAmplitudeNorm2d", SpectralAmplitudeNorm2d)
ul_tasks.__dict__["SpectralAmplitudeNorm2d"] = SpectralAmplitudeNorm2d
if hasattr(ul_tasks, "base_modules") and SpectralAmplitudeNorm2d not in ul_tasks.base_modules:
    ul_tasks.base_modules = ul_tasks.base_modules + (SpectralAmplitudeNorm2d,)

# Extract P3, P4, P5 channel widths from base YOLO11n
base_yolo = YOLO("yolo11n.pt")
base_detect = base_yolo.model.model[-1]
c_p3 = int(base_detect.cv2[0][0].conv.in_channels)
c_p4 = int(base_detect.cv2[1][0].conv.in_channels)
c_p5 = int(base_detect.cv2[2][0].conv.in_channels)

base_cfg = copy.deepcopy(base_yolo.model.yaml)
base_cfg["nc"] = 1
base_cfg["scale"] = "n"
for k in ["yaml_file", "ch"]:
    base_cfg.pop(k, None)

head_layers = base_cfg["head"]
detect_layer = head_layers[-1]
p3_idx, p4_idx, p5_idx = detect_layer[0]
next_idx = len(base_cfg["backbone"]) + len(head_layers) - 1

san_p3 = [p3_idx, 1, "SpectralAmplitudeNorm2d", [c_p3]]
san_p4 = [p4_idx, 1, "SpectralAmplitudeNorm2d", [c_p4]]
san_p5 = [p5_idx, 1, "SpectralAmplitudeNorm2d", [c_p5]]
new_detect = [[next_idx, next_idx + 1, next_idx + 2], 1, "Detect", [1]]

base_cfg["head"] = head_layers[:-1] + [san_p3, san_p4, san_p5, new_detect]

fa_yaml_path = Path("fa_evyolo.yaml")
with open(fa_yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(base_cfg, f, sort_keys=False)

fa_model = YOLO(str(fa_yaml_path))
fa_model.load("yolo11n.pt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fa_model.model.to(device).eval()
with torch.no_grad():
    dummy_input = torch.rand(2, 3, 640, 640, device=device)
    preds = fa_model.model(dummy_input)

print(f"Built fa_evyolo.yaml (P3={c_p3}, P4={c_p4}, P5={c_p5}) | Output shape: {tuple(preds[0].shape)}")

In [25]:
# Cell 4.1: Load or Train FA-EvYOLO with Retinex-FAM Preprocessing Callback
import importlib
from pathlib import Path
import torch
from ultralytics import YOLO
import ultralytics.nn.modules as ul_modules
import ultralytics.nn.tasks as ul_tasks
import fa_evyolo_modules

importlib.reload(fa_evyolo_modules)
from fa_evyolo_modules import SpectralAmplitudeNorm2d, RetinexFourierTransform

setattr(ul_modules, "SpectralAmplitudeNorm2d", SpectralAmplitudeNorm2d)
setattr(ul_tasks, "SpectralAmplitudeNorm2d", SpectralAmplitudeNorm2d)
ul_tasks.__dict__["SpectralAmplitudeNorm2d"] = SpectralAmplitudeNorm2d
if hasattr(ul_tasks, "base_modules"):
    ul_tasks.base_modules = tuple(
        m for m in ul_tasks.base_modules if m.__name__ != "SpectralAmplitudeNorm2d"
    ) + (SpectralAmplitudeNorm2d,)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
source_yaml = OUTPUT_ROOT / "marine_litter" / "marine_litter.yaml"

saved_fa_weights = Path(r"D:\Journal-Project\runs\detect\runs\FA_EvYOLO\train_marine_litter_fa\weights\best.pt")

if not saved_fa_weights.exists():
    print(f"Starting FA-EvYOLO training on {source_yaml}...")
    fa_model = YOLO("fa_evyolo.yaml")
    fa_model.load("yolo11n.pt")

    retinex_fam = RetinexFourierTransform(retinex_weight=0.25, fam_prob=0.50, fam_strength=0.30).to(device)

    def apply_retinex_fam_on_batch(trainer):
        if hasattr(trainer, "batch") and "img" in trainer.batch:
            is_train = trainer.model.training
            trainer.batch["img"] = retinex_fam(trainer.batch["img"], training=is_train)

    fa_model.add_callback("on_preprocess_batch", apply_retinex_fam_on_batch)

    fa_train_results = fa_model.train(
        data=str(source_yaml),
        epochs=100,
        imgsz=640,
        batch=16,
        project="runs/FA_EvYOLO",
        name="train_marine_litter_fa",
        exist_ok=True,
        seed=42,
        deterministic=True,
        verbose=True,
        val=False,
    )
    print("FA-EvYOLO training completed successfully!")
else:
    print(f"Found existing trained 100-epoch FA-EvYOLO checkpoint at:\n{saved_fa_weights}")
    fa_model = YOLO(str(saved_fa_weights))

Found existing trained 100-epoch FA-EvYOLO checkpoint at:
D:\Journal-Project\runs\detect\runs\FA_EvYOLO\train_marine_litter_fa\weights\best.pt


In [26]:
# Cell 4.2: Evaluate Trained FA-EvYOLO Across All 3 Datasets and RSI Glare Strata (Table 1)
weights_dir = Path(r"D:\Journal-Project\runs\detect\runs\FA_EvYOLO\train_marine_litter_fa\weights")
weights_path = weights_dir / "best.pt" if (weights_dir / "best.pt").exists() else weights_dir / "last.pt"

print(f"Loading trained FA-EvYOLO checkpoint from: {weights_path}")
model = YOLO("fa_evyolo.yaml")
model.load(str(weights_path))

source_data = "marine_litter"
results_records = []

for target_data in DATASETS:
    is_in_domain = source_data == target_data
    domain_type = "In-Domain" if is_in_domain else "Out-of-Domain (Zero-Shot)"
    print(f"Evaluating FA-EvYOLO on Target: {target_data.upper()} ({domain_type})...")

    target_yaml = OUTPUT_ROOT / target_data / f"{target_data}.yaml"
    if target_yaml.exists():
        metrics = model.val(data=str(target_yaml), split="test", verbose=False)
        results_records.append({
            "Model": "FA-EvYOLO (Ours)",
            "Train_Source": source_data,
            "Test_Target": target_data,
            "Domain_Shift": domain_type,
            "RSI_Glare_Level": "ALL",
            "mAP_50": round(float(metrics.box.map50), 4),
            "mAP_50_95": round(float(metrics.box.map), 4),
        })

    for rsi in RSI_STRATA:
        rsi_yaml = OUTPUT_ROOT / target_data / f"{target_data}_test_{rsi}.yaml"
        if rsi_yaml.exists():
            rsi_metrics = model.val(data=str(rsi_yaml), split="test", verbose=False)
            results_records.append({
                "Model": "FA-EvYOLO (Ours)",
                "Train_Source": source_data,
                "Test_Target": target_data,
                "Domain_Shift": domain_type,
                "RSI_Glare_Level": rsi,
                "mAP_50": round(float(rsi_metrics.box.map50), 4),
                "mAP_50_95": round(float(rsi_metrics.box.map), 4),
            })

new_df = pd.DataFrame(results_records)
out_csv = OUTPUT_ROOT / "Table1_CrossDataset_RSI_Matrix.csv"

if out_csv.exists():
    old_df = pd.read_csv(out_csv)
    old_df = old_df[old_df["Model"] != "FA-EvYOLO (Ours)"]
    final_df = pd.concat([old_df, new_df], ignore_index=True)
else:
    final_df = new_df

final_df.to_csv(out_csv, index=False)
print("\n--- FINAL Q1 CROSS-DATASET & RSI PERFORMANCE MATRIX (TABLE 1) ---")
print(final_df.to_string(index=False))

Loading trained FA-EvYOLO checkpoint from: D:\Journal-Project\runs\detect\runs\FA_EvYOLO\train_marine_litter_fa\weights\best.pt
WARNING no model scale passed. Assuming scale='n'.


Transferred 547/547 items from pretrained weights
Evaluating FA-EvYOLO on Target: MARINE_LITTER (In-Domain)...
Ultralytics 8.4.182  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
fa_evYOLO summary (fused): 122 layers, 2,690,238 parameters, 105,910 gradients, 6.6 GFLOPs
val: Fast image access  (ping: 0.3±0.2 ms, read: 106.0±66.6 MB/s, size: 48.9 KB)
val: Scanning D:\Journal-Project\Datasets\Q1_Harmonized_Benchmark\marine_litter\test\labels.cache... 348 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 348/348  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 1.2it/s 18.1s0.1s
                   all        348       1571      0.864      0.885      0.925      0.584
Speed: 1.4ms preprocess, 5.0ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to D:\Journal-Project\runs\detect\val-26
Ultralytics 8.4.182  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 

In [14]:
# Cell 5.1: Statistical Calibration Metrics, Letterbox Preprocessing, and IoU Matching Utilities
import torchvision
from scipy.stats import mannwhitneyu

def compute_dece_and_brier(confidences: np.ndarray, is_tp: np.ndarray, n_bins: int = 10) -> tuple:
    if len(confidences) == 0:
        return 0.0, 0.0
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    dece = 0.0
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i + 1])
        prop = np.mean(mask)
        if prop > 0:
            acc = np.mean(is_tp[mask])
            conf = np.mean(confidences[mask])
            dece += abs(conf - acc) * prop
    brier = float(np.mean((confidences - is_tp) ** 2))
    return float(dece), brier


def compute_auroc(y_true_error: np.ndarray, uncertainty_score: np.ndarray) -> float:
    y_true = np.asarray(y_true_error, dtype=np.float64)
    y_score = np.asarray(uncertainty_score, dtype=np.float64)
    n_pos = np.sum(y_true == 1.0)
    n_neg = np.sum(y_true == 0.0)
    if n_pos == 0 or n_neg == 0:
        return 0.5
    order = np.argsort(y_score)
    ranks = np.empty_like(order, dtype=np.float64)
    ranks[order] = np.arange(1, len(y_score) + 1, dtype=np.float64)
    sorted_scores = y_score[order]
    i, n = 0, len(y_score)
    while i < n:
        j = i
        while j + 1 < n and sorted_scores[j + 1] == sorted_scores[i]:
            j += 1
        if j > i:
            ranks[order[i : j + 1]] = 0.5 * (i + 1 + j + 1)
        i = j + 1
    auc = (np.sum(ranks[y_true == 1.0]) - n_pos * (n_pos + 1.0) / 2.0) / (n_pos * n_neg)
    return float(auc)


def compute_aupr(y_true_error: np.ndarray, uncertainty_score: np.ndarray) -> float:
    y_true = np.asarray(y_true_error, dtype=np.float64)
    y_score = np.asarray(uncertainty_score, dtype=np.float64)
    n_pos = np.sum(y_true == 1.0)
    if n_pos == 0:
        return 0.0
    desc = np.argsort(-y_score, kind="mergesort")
    y_sorted = y_true[desc]
    tps = np.cumsum(y_sorted)
    fps = np.cumsum(1.0 - y_sorted)
    prec = tps / np.maximum(tps + fps, 1e-12)
    rec = tps / n_pos
    rec_diff = np.diff(np.concatenate([[0.0], rec]))
    return float(np.sum(prec * rec_diff))


def preprocess_image_and_gt(img_path: Path, label_path: Path, imgsz: int = 640):
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        return None, None
    h0, w0 = img_bgr.shape[:2]
    r = min(imgsz / h0, imgsz / w0)
    new_unpad = (int(round(w0 * r)), int(round(h0 * r)))
    dw, dh = (imgsz - new_unpad[0]) / 2.0, (imgsz - new_unpad[1]) / 2.0

    img_res = cv2.resize(img_bgr, new_unpad, interpolation=cv2.INTER_LINEAR) if (w0, h0) != new_unpad else img_bgr
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    img_pad = cv2.copyMakeBorder(img_res, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(114, 114, 114))

    img_rgb = cv2.cvtColor(img_pad, cv2.COLOR_BGR2RGB)
    img_tensor = torch.from_numpy(img_rgb).permute(2, 0, 1).contiguous().to(device, dtype=torch.float32) / 255.0

    gt_boxes = []
    if label_path.exists():
        for line in label_path.read_text(encoding="utf-8").strip().splitlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                cx, cy, bw, bh = map(float, parts[1:5])
                x1 = (cx - bw / 2.0) * (w0 * r) + left
                y1 = (cy - bh / 2.0) * (h0 * r) + top
                x2 = (cx + bw / 2.0) * (w0 * r) + left
                y2 = (cy + bh / 2.0) * (h0 * r) + top
                gt_boxes.append([x1, y1, x2, y2])

    gt_xyxy = (
        torch.tensor(gt_boxes, dtype=torch.float32, device=device)
        if gt_boxes
        else torch.zeros((0, 4), dtype=torch.float32, device=device)
    )
    return img_tensor, gt_xyxy


def match_tp_fp(pred_boxes: torch.Tensor, gt_boxes: torch.Tensor, iou_thr: float = 0.50) -> np.ndarray:
    n_pred = pred_boxes.shape[0]
    is_tp = np.zeros(n_pred, dtype=np.float64)
    if n_pred == 0 or gt_boxes.shape[0] == 0:
        return is_tp
    ious = torchvision.ops.box_iou(pred_boxes, gt_boxes).cpu().numpy()
    matched_gt = set()
    for i in range(n_pred):
        best_gt = -1
        best_iou = iou_thr
        for j in range(gt_boxes.shape[0]):
            if j not in matched_gt and ious[i, j] >= best_iou:
                best_iou = ious[i, j]
                best_gt = j
        if best_gt != -1:
            is_tp[i] = 1.0
            matched_gt.add(best_gt)
    return is_tp

print("Cell 5.1 executed: Calibration metrics and IoU matching utilities ready.")

Cell 5.1 executed: Calibration metrics and IoU matching utilities ready.


In [15]:
# Cell 5.2: Run Multi-Dataset Evidential Calibration & Pillar Verification (Table 2)
import torch.nn.functional as F
from fa_evyolo_modules import EvidentialRegularizer, ReflectionUncertaintyNMS

weights_dir = Path(r"D:\Journal-Project\runs\detect\runs\FA_EvYOLO\train_marine_litter_fa\weights")
fa_weights = weights_dir / "best.pt" if (weights_dir / "best.pt").exists() else weights_dir / "last.pt"

fa_yolo = YOLO("fa_evyolo.yaml")
fa_yolo.load(str(fa_weights))
fa_net = fa_yolo.model.to(device).eval()

detect_head = fa_net.model[-1]
cv2_buffers, cv3_buffers = [], []

for scale_idx in range(3):
    detect_head.cv2[scale_idx]._forward_hooks.clear()
    detect_head.cv3[scale_idx]._forward_hooks.clear()
    detect_head.cv2[scale_idx].register_forward_hook(lambda m, inp, out: cv2_buffers.append(out))
    detect_head.cv3[scale_idx].register_forward_hook(lambda m, inp, out: cv3_buffers.append(out))

ru_nms = ReflectionUncertaintyNMS(gamma_u=0.35, gamma_g=0.65, iou_thres=0.65)
records = []

for ds_name in DATASETS:
    print(f"Evaluating uncertainty calibration and RU-NMS on: {ds_name.upper()}...")
    img_dir = OUTPUT_ROOT / ds_name / "test" / "images"
    lbl_dir = OUTPUT_ROOT / ds_name / "test" / "labels"
    img_files = sorted(list(img_dir.glob("*.*")))

    raw_confs_all, ev_confs_all = [], []
    is_tp_raw_all, is_tp_runms_all = [], []
    u_cls_all, u_loc_all, u_joint_all = [], [], []

    with torch.no_grad():
        for img_path in img_files:
            lbl_path = lbl_dir / f"{img_path.stem}.txt"
            img_tensor, gt_xyxy = preprocess_image_and_gt(img_path, lbl_path)
            if img_tensor is None:
                continue

            cv2_buffers.clear()
            cv3_buffers.clear()
            preds = fa_net(img_tensor.unsqueeze(0))
            pred_tensor = preds[0][0]

            dfl_flat = torch.cat([t.flatten(2) for t in cv2_buffers], dim=2)[0].permute(1, 0).to(torch.float32)
            cls_flat = torch.cat([t.flatten(2) for t in cv3_buffers], dim=2)[0].permute(1, 0).to(torch.float32)

            scores = pred_tensor[4]
            cand_mask = scores >= 0.15
            if not cand_mask.any():
                continue

            cx, cy = pred_tensor[0, cand_mask], pred_tensor[1, cand_mask]
            bw, bh = pred_tensor[2, cand_mask], pred_tensor[3, cand_mask]
            boxes_xyxy = torch.stack([cx - 0.5 * bw, cy - 0.5 * bh, cx + 0.5 * bw, cy + 0.5 * bh], dim=-1)
            cand_scores = scores[cand_mask].to(torch.float32)
            cand_cls = cls_flat[cand_mask, 0]
            cand_dfl = dfl_flat[cand_mask]

            unc_dict = EvidentialRegularizer.extract_box_uncertainty(cand_cls, cand_dfl, reg_max=16)
            u_cls = unc_dict["u_cls"]
            u_loc = unc_dict["u_loc"]
            u_total = unc_dict["u_total"]

            # Standard NMS Baseline
            keep_std = torchvision.ops.nms(boxes_xyxy, cand_scores, 0.65)
            std_boxes = boxes_xyxy[keep_std]
            std_confs = cand_scores[keep_std].cpu().numpy()
            tp_std = match_tp_fp(std_boxes, gt_xyxy, iou_thr=0.50)

            raw_confs_all.append(std_confs)
            is_tp_raw_all.append(tp_std)

            # Pillar 3: Reflection-Uncertainty Modulated NMS (RU-NMS)
            glare_density = ru_nms.compute_box_glare_density(img_tensor, boxes_xyxy)
            mod_boxes, mod_scores, keep_ru = ru_nms.modulate_and_filter(
                boxes_xyxy, cand_scores, u_total, img_tensor, conf_thres=0.0
            )
            tp_ru = match_tp_fp(mod_boxes, gt_xyxy, iou_thr=0.50)

            ev_confs_all.append(mod_scores.cpu().numpy())
            is_tp_runms_all.append(tp_ru)

            u_cls_all.append(u_cls[keep_ru].cpu().numpy())
            u_loc_all.append(u_loc[keep_ru].cpu().numpy())
            u_joint_all.append((u_total[keep_ru] * (1.0 + 0.65 * glare_density[keep_ru])).cpu().numpy())

    raw_confs = np.concatenate(raw_confs_all)
    is_tp_raw = np.concatenate(is_tp_raw_all)
    ev_confs = np.concatenate(ev_confs_all)
    is_tp_ru = np.concatenate(is_tp_runms_all)
    is_fp_ru = 1.0 - is_tp_ru

    u_cls_vec = np.concatenate(u_cls_all)
    u_joint_vec = np.concatenate(u_joint_all)

    dece_uncal, brier_uncal = compute_dece_and_brier(raw_confs, is_tp_raw)
    dece_ev, brier_ev = compute_dece_and_brier(ev_confs, is_tp_ru)

    auroc_cls = compute_auroc(is_fp_ru, u_cls_vec)
    auroc_joint = compute_auroc(is_fp_ru, u_joint_vec)
    aupr_joint = compute_aupr(is_fp_ru, u_joint_vec)

    u_tp_mean = float(np.mean(u_joint_vec[is_tp_ru == 1.0])) if np.any(is_tp_ru == 1.0) else 0.0
    u_fp_mean = float(np.mean(u_joint_vec[is_fp_ru == 1.0])) if np.any(is_fp_ru == 1.0) else 0.0
    _, p_val = mannwhitneyu(u_joint_vec[is_fp_ru == 1.0], u_joint_vec[is_tp_ru == 1.0], alternative="greater")

    records.append({
        "Dataset": ds_name,
        "Domain": "In-Domain" if ds_name == "marine_litter" else "Zero-Shot OOD",
        "Uncal_ECE (Down)": round(dece_uncal, 4),
        "Evidential_D-ECE (Down)": round(dece_ev, 4),
        "Uncal_Brier (Down)": round(brier_uncal, 4),
        "Evidential_Brier (Down)": round(brier_ev, 4),
        "AUROC_u_cls (Up)": round(auroc_cls, 4),
        "AUROC_Full_Ev+Glare (Up)": round(auroc_joint, 4),
        "AUPR_Error_Detect (Up)": round(aupr_joint, 4),
        "Mean_u (TP)": round(u_tp_mean, 4),
        "Mean_u (FP/Glare)": round(u_fp_mean, 4),
        "MannWhitney_p": f"{p_val:.2e}",
    })

for scale_idx in range(3):
    detect_head.cv2[scale_idx]._forward_hooks.clear()
    detect_head.cv3[scale_idx]._forward_hooks.clear()

df_table2 = pd.DataFrame(records)
table2_csv = OUTPUT_ROOT / "Table2_Uncertainty_Calibration_and_Pillar_Verification.csv"
df_table2.to_csv(table2_csv, index=False)

print(f"\nSaved Table 2 to: {table2_csv}")
print("\n--- TABLE 2: EVIDENTIAL CALIBRATION (D-ECE) & ERROR-DETECTION VERIFICATION ---")
print(df_table2.to_string(index=False))

WARNING no model scale passed. Assuming scale='n'.
Transferred 547/547 items from pretrained weights
Evaluating uncertainty calibration and RU-NMS on: MARINE_LITTER...
Evaluating uncertainty calibration and RU-NMS on: FLOATING_WASTE...
Evaluating uncertainty calibration and RU-NMS on: FLOW_IMG...

Saved Table 2 to: D:\Journal-Project\Datasets\Q1_Harmonized_Benchmark\Table2_Uncertainty_Calibration_and_Pillar_Verification.csv

--- TABLE 2: EVIDENTIAL CALIBRATION (D-ECE) & ERROR-DETECTION VERIFICATION ---
       Dataset        Domain  Uncal_ECE (Down)  Evidential_D-ECE (Down)  Uncal_Brier (Down)  Evidential_Brier (Down)  AUROC_u_cls (Up)  AUROC_Full_Ev+Glare (Up)  AUPR_Error_Detect (Up)  Mean_u (TP)  Mean_u (FP/Glare) MannWhitney_p
 marine_litter     In-Domain            0.1260                   0.1975              0.1309                   0.1582            0.8869                    0.8808                  0.7258       0.3528             0.5491      0.00e+00
floating_waste Zero-Shot OOD  